# SFT Training — QLoRA Fine-Tuning of Qwen2.5-3B-Instruct

Fine-tune `Qwen/Qwen2.5-3B-Instruct` on the structured SFT dataset  
(`sft_execution_data.jsonl`) using **QLoRA** (4-bit NF4 + LoRA) via `trl.SFTTrainer`.

Each training example is a 3-turn conversation (ChatML format):
- **system**: instructs the model to apply a specific ICR strategy
- **user**: question + documents + strategy
- **assistant**: strict JSON with `strategy`, `response`, `justification`

The fine-tuned model will serve as the base for GRPO (RLVR) training.

## 1. Runtime Parameters

In [ ]:
from pathlib import Path

REPO_URL    = "https://gitlab.com/beryl.hoe/arc.git"
PROJECT_DIR = Path("/content/arc")
DATA_PATH   = PROJECT_DIR / "data" / "sft_execution_data.jsonl"
OUTPUT_DIR  = PROJECT_DIR / "models" / "sft_model"

# Base model
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# QLoRA — quantization
LOAD_IN_4BIT        = True
BNB_4BIT_QUANT_TYPE = "nf4"
BNB_COMPUTE_DTYPE   = "float16"   # "bfloat16" on Ampere+ GPUs (A100, H100)

# LoRA adapters
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training
LEARNING_RATE               = 2e-5
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_TRAIN_EPOCHS            = 3
MAX_SEQ_LENGTH              = 2048
WARMUP_RATIO                = 0.05
LR_SCHEDULER_TYPE           = "cosine"
LOGGING_STEPS               = 10
SAVE_STEPS                  = 100
FP16                        = True   # set False + BF16=True on Ampere+
BF16                        = False

# Fraction of examples set aside for eval
VAL_SPLIT = 0.05

# Hugging Face token (only needed for gated models)
HF_TOKEN = ""   # or export HF_TOKEN in the environment

FORCE_RECLONE = False

## 2. Install System Packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y git git-lfs wget > /dev/null
!git lfs install

## 3. Install Python Dependencies

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "trl>=0.9.0",
    "peft>=0.11.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.30.0",
    "transformers>=4.41.0",
    "datasets>=2.20.0",
    "torch",
    "matplotlib",
], check=True)
print("Dependencies installed")

## 4. Clone the Repository

In [ ]:
import shutil, subprocess

if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if PROJECT_DIR.exists():
    print(f"Repository already exists at {PROJECT_DIR}; pulling latest changes.")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

print(f"Project directory: {PROJECT_DIR}")

## 5. Optional Hugging Face Authentication

In [ ]:
import os

hf_token = HF_TOKEN or os.environ.get("HF_TOKEN", "")
if hf_token:
    subprocess.run(["huggingface-cli", "login", "--token", hf_token], check=True)
    print("Logged in to Hugging Face")
else:
    print("No HF_TOKEN found — skipping (only needed for gated models)")

## 6. Load & Inspect Dataset

In [ ]:
import json, re
from collections import Counter

print(f"Loading dataset from {DATA_PATH} ...")
raw_records = []
with DATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_records.append(json.loads(line))

print(f"Total examples: {len(raw_records)}")

# Strategy distribution
strategy_counter = Counter()
_pat = re.compile('\"strategy\"\s*:\s*\"(S\d)\"')
for rec in raw_records:
    m = _pat.search(rec["messages"][2]["content"])
    if m:
        strategy_counter[m.group(1)] += 1

print("Distribution by strategy:")
for s, c in sorted(strategy_counter.items()):
    print(f"  {s}: {c}")

# Show first example
first = raw_records[0]
print("--- First example ---")
for msg in first["messages"]:
    print("[" + msg["role"].upper() + "]")
    content = msg["content"]
    print(content[:400] + ("..." if len(content) > 400 else ""))

## 7. Build Hugging Face Dataset & Train/Eval Split

In [ ]:
from datasets import Dataset
import random

random.seed(42)
random.shuffle(raw_records)

n_val   = max(1, int(len(raw_records) * VAL_SPLIT))
n_train = len(raw_records) - n_val

train_records = raw_records[:n_train]
val_records   = raw_records[n_train:]

train_dataset = Dataset.from_list(train_records)
val_dataset   = Dataset.from_list(val_records)

print(f"Train: {len(train_dataset)} examples")
print(f"Val  : {len(val_dataset)} examples")

## 8. Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)
tokenizer.padding_side = "right"   # SFTTrainer expects right-padding
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded — vocab size: {tokenizer.vocab_size}")
print(f"  Pad token : {tokenizer.pad_token!r}")
print(f"  EOS token : {tokenizer.eos_token!r}")
print(f"  Chat template present: {tokenizer.chat_template is not None}")

## 9. Define Chat Formatting Function

`SFTTrainer` with `formatting_func` expects a function that receives a batch  
dict (column-of-lists) and returns a list of strings.  
We apply Qwen's chat template so the model learns from properly formatted turns.

In [ ]:
def format_chat(example_or_batch: dict):
    """Apply Qwen chat template to a single example or a batch."""
    messages = example_or_batch["messages"]
    if len(messages) > 0 and isinstance(messages[0], list):
        # It's a batch
        return [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
            for m in messages
        ]
    else:
        # It's a single example
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

# Quick sanity check
sample_text = format_chat(train_dataset[0])
if isinstance(sample_text, list): sample_text = sample_text[0]
print("--- Rendered example (first 600 chars) ---")
print(sample_text[:600])
print(f"Total length (chars): {len(sample_text)}")

## 10. Load Base Model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if BNB_COMPUTE_DTYPE == 'bfloat16' else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,  # nested quantization saves ~0.4 bit/param
)

print(f"Loading model: {MODEL_NAME} in 4-bit ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False      # required for gradient checkpointing
model.config.pretraining_tp = 1     # single-GPU: no tensor parallelism

print("Model loaded")
total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_params / 1e9:.2f}B")

## 11. Configure LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Cast layer norms and LM head to fp32 for training stability
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 12. Training Arguments

In [ ]:
from transformers import TrainingArguments

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,
    fp16=FP16,
    bf16=BF16,
    logging_dir=str(OUTPUT_DIR / 'logs'),
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",      # set "wandb" or "tensorboard" if needed
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",   # memory-efficient optimizer from bitsandbytes
    dataloader_pin_memory=False,
)

print("TrainingArguments configured")
eff_batch = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
print(f"  Effective batch size: {eff_batch}")

## 13. Build SFTTrainer

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    formatting_func=format_chat,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
)

print("SFTTrainer ready")
steps_per_epoch = len(train_dataset) // (PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total steps    : {steps_per_epoch * NUM_TRAIN_EPOCHS}")

## 14. Train

In [ ]:
import time

print("Starting SFT training ...")
t0 = time.time()

train_result = trainer.train()

elapsed = time.time() - t0
print(f"Training complete in {elapsed / 60:.1f} min")
print(f"  Final train loss: {train_result.training_loss:.4f}")

## 15. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps  = [e["step"] for e in log_history if "loss" in e]
train_losses = [e["loss"] for e in log_history if "loss" in e]
eval_steps   = [e["step"] for e in log_history if "eval_loss" in e]
eval_losses  = [e["eval_loss"] for e in log_history if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_steps, train_losses, label="Train loss", linewidth=1.5)
if eval_losses:
    ax.plot(eval_steps, eval_losses, "o-", label="Eval loss", linewidth=1.5)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("SFT Training — Cross-Entropy Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'loss_curve.png'), dpi=150)
plt.show()
print(f"Loss curve saved to {OUTPUT_DIR / 'loss_curve.png'}")

## 16. Save LoRA Adapter Weights

In [ ]:
print(f"Saving LoRA adapter weights to {OUTPUT_DIR} ...")
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print("Saved:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}")

## 17. (Optional) Merge LoRA into Base Model

Produces a standalone model that works without PEFT.  
Requires enough RAM / VRAM for the full fp16 model (~6 GB).

In [ ]:
MERGE_AND_SAVE = True   # set False to skip

if MERGE_AND_SAVE:
    from peft import AutoPeftModelForCausalLM

    merged_dir = OUTPUT_DIR.parent / 'sft_model_merged'
    merged_dir.mkdir(parents=True, exist_ok=True)

    print(f"Merging LoRA weights into base model -> {merged_dir}")
    merged_model = AutoPeftModelForCausalLM.from_pretrained(
        str(OUTPUT_DIR),
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    merged_model = merged_model.merge_and_unload()
    merged_model.save_pretrained(str(merged_dir), safe_serialization=True)
    tokenizer.save_pretrained(str(merged_dir))
    print(f"Merged model saved to {merged_dir}")
else:
    print("Skipping merge step")

## 18. Validation Inference

Generate one response from the fine-tuned model and verify it contains  
valid JSON with `strategy`, `response`, and `justification`.

In [ ]:
import json, re
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

print("Loading adapter model for inference check ...")
inf_tokenizer = AutoTokenizer.from_pretrained(str(OUTPUT_DIR), trust_remote_code=True)
inf_model = AutoPeftModelForCausalLM.from_pretrained(
    str(OUTPUT_DIR),
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
inf_model.eval()

# Use system + user turns from the first validation example
val_example     = val_dataset[0]
prompt_messages = val_example["messages"][:2]   # system + user only

prompt_text = inf_tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = inf_tokenizer(prompt_text, return_tensors='pt').to(inf_model.device)

with torch.no_grad():
    outputs = inf_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=inf_tokenizer.pad_token_id,
        eos_token_id=inf_tokenizer.eos_token_id,
    )

prompt_len    = inputs['input_ids'].shape[1]
generated_ids = outputs[0][prompt_len:]
response_text = inf_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("--- Model response ---")
print(response_text[:600])

# Validate JSON structure
print("--- JSON validation ---")
clean = re.sub(
    r"^```json\s*|^```\s*|```$",
    "",
    response_text.strip(),
    flags=re.MULTILINE,
).strip()
try:
    parsed = json.loads(clean)
    required = {"strategy", "response", "justification"}
    missing  = required - parsed.keys()
    if missing:
        print(f"Missing keys: {missing}")
    else:
        print("Valid JSON with all required fields!")
        print(json.dumps(parsed, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print(f"  Raw text: {response_text[:200]}")

# Compare to ground-truth
expected = val_example["messages"][2]["content"]
print("--- Expected (gold) ---")
print(expected[:400])

## 19. Summary

| Component | Value |
|-----------|-------|
| Base model | `Qwen/Qwen2.5-3B-Instruct` |
| Quantization | 4-bit NF4 (QLoRA) |
| LoRA rank / alpha | 16 / 32 |
| Target modules | q_proj, k_proj, v_proj, o_proj |
| Train / Val examples | ~1,392 / ~74 |
| Effective batch | 16 |
| Max sequence length | 2,048 tokens |
| Epochs | 3 |
| Optimizer | paged_adamw_8bit |
| LR scheduler | cosine (warmup 5 %) |
| LoRA adapter output | `./models/sft_model` |
| Merged model | `./models/sft_model_merged` |

**Next step:** Use `sft_model_merged` as the base policy for GRPO training.